In [ ]:
import os
import numpy as np
import nibabel as nib

source_path = r"D:\TEKNIK INFORMATIKA\SKRIPSI\Kagayaki\ImageCHD_dataset"
nnunet_root = r"D:\TEKNIK INFORMATIKA\SKRIPSI\Kagayaki\nnUNet_raw"
dataset_name = "Dataset001_ImageCHD"
TF = 256

dataset_path = os.path.join(nnunet_root, dataset_name)
imagesTr = os.path.join(dataset_path, "imagesTr")
labelsTr = os.path.join(dataset_path, "labelsTr")

os.makedirs(imagesTr, exist_ok=True)
os.makedirs(labelsTr, exist_ok=True)

In [2]:
def frame_generalization(volume):
    H, W, F = volume.shape

    print(f"Original frame = {F}")

    if F == TF:
        return volume

    # Symmetric Trimming
    if F > TF:
        N = F - TF

        if N % 2 == 0:
            first = N // 2
            last = N // 2
        else:
            first = (N // 2) + 1
            last = N // 2

        print(f"Trimming: first={first}, last={last}")

        return volume[:, :, first:F-last]

    # Duplication with Indexing
    N = TF - F
    print(f"Duplication needed = {N}")

    # SINGLE DUPLICATION
    if N < TF / 2:
        i = (F - 1) // 2

        if N % 2 == 0:
            left = N // 2
            right = N // 2
        else:
            left = (N // 2) + 1
            right = N // 2

        start = max(0, i - left)
        end = min(F - 1, i + right)

        print(f"Duplicate range = {start} to {end}")

        new_frames = []

        for idx in range(F):
            new_frames.append(volume[:, :, idx])
            if start <= idx <= end:
                new_frames.append(volume[:, :, idx])

        new_volume = np.stack(new_frames, axis=2)

        if new_volume.shape[2] > TF:
            new_volume = new_volume[:, :, :TF]

        return new_volume

    # DOUBLE DUPLICATION
    print("Double duplication mode")

    new_frames = []
    for idx in range(F):
        new_frames.append(volume[:, :, idx])
        new_frames.append(volume[:, :, idx])

    volume_2 = np.stack(new_frames, axis=2)
    F2 = volume_2.shape[2]

    N2 = TF - F2

    if N2 <= 0:
        return volume_2[:, :, :TF]

    i = (F2 - 1) // 2

    if N2 % 2 == 0:
        left = N2 // 2
        right = N2 // 2
    else:
        left = (N2 // 2) + 1
        right = N2 // 2

    start = max(0, i - left)
    end = min(F2 - 1, i + right)

    final_frames = []

    for idx in range(F2):
        final_frames.append(volume_2[:, :, idx])
        if start <= idx <= end:
            final_frames.append(volume_2[:, :, idx])

    final_volume = np.stack(final_frames, axis=2)

    if final_volume.shape[2] > TF:
        final_volume = final_volume[:, :, :TF]

    return final_volume

In [3]:
files = sorted([f for f in os.listdir(source_path) if f.endswith("_image.nii.gz")])

for img_file in files:

    patient_id = img_file.split("_")[1]  
    label_file = f"ct_{patient_id}_label.nii.gz"

    image_path = os.path.join(source_path, img_file)
    label_path = os.path.join(source_path, label_file)

    if not os.path.exists(label_path):
        print(f"Missing label for patient {patient_id}")
        continue

    print(f"\n=== Patient {patient_id} ===")

    img = nib.load(image_path)
    lbl = nib.load(label_path)

    img_data = img.get_fdata()
    lbl_data = lbl.get_fdata()

    new_img = frame_generalization(img_data)
    new_lbl = frame_generalization(lbl_data)

    # nnU-Net format
    image_dst = os.path.join(imagesTr, f"{patient_id}_0000.nii.gz")
    label_dst = os.path.join(labelsTr, f"{patient_id}.nii.gz")

    nib.save(nib.Nifti1Image(new_img, img.affine, img.header), image_dst)
    nib.save(nib.Nifti1Image(new_lbl, lbl.affine, lbl.header), label_dst)

    print(f"Saved as {patient_id}_0000.nii.gz")


=== Patient 1001 ===
Original frame = 221
Duplication needed = 35
Duplicate range = 92 to 127
Original frame = 221
Duplication needed = 35
Duplicate range = 92 to 127
Saved as 1001_0000.nii.gz

=== Patient 1002 ===
Original frame = 137
Duplication needed = 119
Duplicate range = 8 to 127
Original frame = 137
Duplication needed = 119
Duplicate range = 8 to 127
Saved as 1002_0000.nii.gz

=== Patient 1003 ===
Original frame = 206
Duplication needed = 50
Duplicate range = 77 to 127
Original frame = 206
Duplication needed = 50
Duplicate range = 77 to 127
Saved as 1003_0000.nii.gz

=== Patient 1004 ===
Original frame = 344
Trimming: first=44, last=44
Original frame = 344
Trimming: first=44, last=44
Saved as 1004_0000.nii.gz

=== Patient 1005 ===
Original frame = 275
Trimming: first=10, last=9
Original frame = 275
Trimming: first=10, last=9
Saved as 1005_0000.nii.gz

=== Patient 1007 ===
Original frame = 275
Trimming: first=10, last=9
Original frame = 275
Trimming: first=10, last=9
Saved as 1